# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/) available at the FAIR² initiative, providing detailed record structure and metadata.

In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and display high-level summary.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Obtain metadata as a dictionary
metadata = dataset.metadata.to_json()

print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Published: {metadata['datePublished']}")
print(f"Authors: {[a['@id'] for a in metadata['author']]}")

## 2. Data Overview
List the available Record Sets, their `@id`, available fields with their `@id`s, and broader schema structure.

In [ ]:
# Show all Record Sets in the dataset
print("Available Record Sets (by @id):")
record_sets = dataset.record_sets()
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', '<unnamed>')})")

# For each record set, print its available fields and columns
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    # Fields
    print("  Fields:")
    for field in rs.get('field', []):
        # If fields are objects, print id and name
        if isinstance(field, dict):
            print(f"    - {field.get('@id', field)} (name: {field.get('name', '<unnamed>')})")
        else:
            print(f"    - {field}")
    # Columns
    print("  Columns in file(s):")
    for fileobj in rs.get('fileObject', []):
        print(f"    File: {fileobj.get('@id', fileobj)}")
        for column in fileobj.get('column', []):
            if isinstance(column, dict):
                print(f"      - {column.get('@id', column)} (name: {column.get('name', '<unnamed>')})")
            else:
                print(f"      - {column}")

## 3. Data Extraction
Select desired record set (by its `@id`) and load it into a DataFrame. You can use the printed record set and field `@id`s from the cell above.

In [ ]:
# We'll select the main record set, which often has an @id like 'cr:RecordSet' or similar.
# Let us find the main record set id again for clarity.
record_sets = dataset.record_sets()

# In this dataset, there may be one primary record set.
if record_sets:
    main_record_set = record_sets[0]['@id']
    print(f"Using Record Set: {main_record_set}")
else:
    raise ValueError("No record sets present in the dataset.")

# Extract data from this record set
records = list(dataset.records(record_set=main_record_set))
df = pd.DataFrame(records)

print(f"Fields in DataFrame:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group by relevant fields. Here we use the field `cr:peptideCounts` as an example numeric field, and `cr:accession` as a (likely) categorical field; replace these with actual field `@id`s/column names as needed based on the dataset's real schema.

In [ ]:
# Set your numeric field and grouping field based on data overview above
# You may have e.g. 'cr:peptideCounts', 'cr:abundance', 'cr:coveragePercent', etc.
# Let's print the columns again for selection
print("Available columns:", df.columns.tolist())

# We'll try to use common possible field keys. Update these as necessary for your specific dataset.
numeric_field_id = 'cr:peptideCounts' if 'cr:peptideCounts' in df.columns else df.select_dtypes(include='number').columns[0]
group_field_id = 'cr:accession' if 'cr:accession' in df.columns else df.columns[0]  # Use the main categorical field
print(f"Numeric Field: {numeric_field_id}")
print(f"Grouping Field: {group_field_id}")

# Filter records where the numeric field is above a threshold
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the chosen numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by grouping field and calculate mean (remove non-numeric columns for groupby)
grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
print(f"\nGrouped data by {group_field_id} (mean of numeric fields):")
print(grouped_df.head())

## 5. Visualization
Plot histograms and relationships for the main numeric field(s).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the (filtered and normalized) numeric field
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id} (z-score normalized, filtered > {threshold})")
plt.xlabel(f"{numeric_field_id} (normalized)")
plt.show()

# Scatter plot of numeric_value vs. another numeric if present
numeric_columns = filtered_df.select_dtypes(include='number').columns.tolist()
candidate = [col for col in numeric_columns if col != numeric_field_id and col != f"{numeric_field_id}_normalized"]
if candidate:
    scatter_field = candidate[0]
    plt.figure(figsize=(7,5))
    sns.scatterplot(x=filtered_df[numeric_field_id], y=filtered_df[scatter_field])
    plt.title(f"{numeric_field_id} vs. {scatter_field}")
    plt.xlabel(numeric_field_id)
    plt.ylabel(scatter_field)
    plt.show()
else:
    print('Could not find a second numeric field for scatter plot.')

## 6. Conclusion
We loaded the FAIR² mass spectrometry dataset using `mlcroissant`, explored record structures via their `@id`, extracted the core table, filtered and normalized numeric fields, and visualized distributions.

You can now further explore relationships between other fields, perform more advanced statistics, or integrate this data into downstream analyses! Review the dataset schema for detailed field semantics and refer to the Croissant documentation for additional mlcroissant features.